In [ ]:
import pandas as pd
df = pd.read_csv("sim.csv")

In [ ]:
from collections import defaultdict
# Keep only necessary columns to reduce memory
df = df[['nameOrig', 'nameDest', 'amount', 'isFraud']]

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (6362620, 4)
      nameOrig     nameDest    amount  isFraud
0  C1231006815  M1979787155   9839.64        0
1  C1666544295  M2044282225   1864.28        0
2  C1305486145   C553264065    181.00        1
3   C840083671    C38997010    181.00        1
4  C2048537720  M1230701703  11668.14        0


In [ ]:
# Get all unique accounts
accounts = sorted(set(df['nameOrig']) | set(df['nameDest']))

# Map account IDs to indices (required for Union-Find)
account_to_idx = {acc: i for i, acc in enumerate(accounts)}

print("Total unique accounts:", len(accounts))

Total unique accounts: 9073900


In [ ]:
graph = defaultdict(list)

for _, row in df.iterrows():
    sender = row['nameOrig']
    receiver = row['nameDest']
    graph[sender].append(receiver)

print("Graph built with nodes:", len(graph))

Graph built with nodes: 6353307


In [ ]:
class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))
        self.rank = [0] * size
        self.size = [1] * size

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # Path compression
        return self.parent[x]

    def union(self, x, y):
        rootX = self.find(x)
        rootY = self.find(y)

        if rootX == rootY:
            return

        # Union by rank
        if self.rank[rootX] < self.rank[rootY]:
            self.parent[rootX] = rootY
            self.size[rootY] += self.size[rootX]
        elif self.rank[rootX] > self.rank[rootY]:
            self.parent[rootY] = rootX
            self.size[rootX] += self.size[rootY]
        else:
            self.parent[rootY] = rootX
            self.rank[rootX] += 1
            self.size[rootX] += self.size[rootY]

In [ ]:
uf = UnionFind(len(accounts))

for _, row in df.iterrows():
    sender = row['nameOrig']
    receiver = row['nameDest']

    idx1 = account_to_idx[sender]
    idx2 = account_to_idx[receiver]

    uf.union(idx1, idx2)

In [ ]:

components = defaultdict(list)

for acc in accounts:
    idx = account_to_idx[acc]
    root = uf.find(idx)
    components[root].append(acc)

print("Total clusters detected:", len(components))

Total clusters detected: 2711280


In [ ]:
# Sort clusters by size
cluster_sizes = sorted(
    [(root, len(members)) for root, members in components.items()],
    key=lambda x: x[1],
    reverse=True
)

print("\nTop 5 Largest Clusters:")
for i, (root, size) in enumerate(cluster_sizes[:5], 1):
    print(f"{i}. Cluster Root {root}: {size} accounts")

In [ ]:
# Map account -> fraud count
fraud_accounts = set()

for _, row in df[df['isFraud'] == 1].iterrows():
    fraud_accounts.add(row['nameOrig'])
    fraud_accounts.add(row['nameDest'])

print("Total fraud-related accounts:", len(fraud_accounts))

Total fraud-related accounts: 16382


In [ ]:
print("\nClusters containing fraud accounts:")

for root, members in components.items():
    fraud_in_cluster = [acc for acc in members if acc in fraud_accounts]

    if len(fraud_in_cluster) > 0:
        print(f"Cluster {root}: Size={len(members)}, Fraud Accounts={len(fraud_in_cluster)}")


Clusters containing fraud accounts:
Cluster 103: Size=2, Fraud Accounts=2
Cluster 2378744: Size=4, Fraud Accounts=2
Cluster 4343425: Size=49, Fraud Accounts=2
Cluster 3544244: Size=11, Fraud Accounts=2
Cluster 5519991: Size=19, Fraud Accounts=2
Cluster 6747855: Size=18, Fraud Accounts=2
Cluster 3152019: Size=30, Fraud Accounts=2
Cluster 288: Size=10, Fraud Accounts=2
Cluster 4712104: Size=11, Fraud Accounts=2
Cluster 4366993: Size=16, Fraud Accounts=2
Cluster 4293506: Size=30, Fraud Accounts=2
Cluster 1385988: Size=11, Fraud Accounts=2
Cluster 3570319: Size=19, Fraud Accounts=2
Cluster 4456515: Size=31, Fraud Accounts=2
Cluster 4150870: Size=10, Fraud Accounts=2
Cluster 707433: Size=30, Fraud Accounts=2
Cluster 6046120: Size=22, Fraud Accounts=2
Cluster 1160: Size=2, Fraud Accounts=2
Cluster 1795832: Size=2, Fraud Accounts=2
Cluster 2250931: Size=11, Fraud Accounts=2
Cluster 1982329: Size=8, Fraud Accounts=2
Cluster 3466233: Size=2, Fraud Accounts=2
Cluster 2244796: Size=16, Fraud Acc